# 3. YOLOv3 - One-Stage Object Detection

## Buoc chuyen tu Two-Stage sang One-Stage

### YOLOv1 (2016) - Y tuong dot pha
- **Core idea**: Chia image thanh S x S grid, moi cell predict B bounding boxes + confidence + class probabilities
- **Uu diem**: Cuc nhanh (45 FPS), nhin toan bo image (global context)
- **Nhuoc diem**: Kho detect small objects, gioi han 2 boxes/cell, low recall
- **mAP**: 63.4% tren VOC 2007

### YOLOv2 / YOLO9000 (2017)
- Cai tien: Batch Normalization, Anchor Boxes, Multi-scale training, Darknet-19 backbone
- Passthrough layer de giu thong tin spatial

### YOLOv3 (2018) - Model chung ta cai dat
- **Backbone**: Darknet-53 (53 convolutional layers, su dung residual connections)
- **Multi-scale Detection**: Predict o 3 scales khac nhau (13x13, 26x26, 52x52)
- **Anchor Boxes**: 9 anchor boxes (3 per scale), duoc tinh bang k-means clustering
- **Loss**: Binary cross-entropy cho class predictions (multi-label, khong dung softmax)

## Kien truc YOLOv3

```
Input (416x416)
      |
[Darknet-53 Backbone]
      |
      ├── Scale 1 (13x13) ---> Detect large objects
      |       |
      |   [Upsample + Concat]
      |       |
      ├── Scale 2 (26x26) ---> Detect medium objects
      |       |
      |   [Upsample + Concat]
      |       |
      └── Scale 3 (52x52) ---> Detect small objects

Moi scale predict: B x (5 + C) per grid cell
  - B = 3 anchor boxes per scale
  - 5 = tx, ty, tw, th, objectness
  - C = num_classes (20 cho VOC)
```

## So sanh voi Faster R-CNN
| | Faster R-CNN | YOLOv3 |
|---|---|---|
| Approach | Two-stage (RPN + Head) | One-stage (single pass) |
| Speed | ~5 FPS | ~30 FPS |
| Small objects | Tot (RoI Align) | Kha (multi-scale) |
| Global context | Khong (chi nhin region) | Co (nhin toan bo image) |

In [ ]:
import sys
sys.path.append("../src")

import os
import time
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

%matplotlib inline

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

DATA_ROOT = "../data"
RESULTS_DIR = "../results/yolov3"
os.makedirs(RESULTS_DIR, exist_ok=True)

## 3.1 Su dung Ultralytics YOLOv3

Ultralytics ho tro YOLOv3 (yolov3.pt, yolov3-tiny.pt). Ta se:
1. Load pretrained YOLOv3 (train tren COCO)
2. Fine-tune tren PASCAL VOC 2012
3. Danh gia va so sanh voi Faster R-CNN

In [ ]:
from ultralytics import YOLO

# Load YOLOv3 pretrained tren COCO
model = YOLO("yolov3u.pt")  # YOLOv3 Ultralytics version

# In thong tin model
print(f"Model: YOLOv3u")
print(f"Task: {model.task}")
print(f"Number of parameters: {sum(p.numel() for p in model.model.parameters()):,}")

## 3.2 Chuan bi Dataset cho YOLO format

YOLO yeu cau dataset theo format rieng:
```
dataset/
├── images/
│   ├── train/
│   └── val/
├── labels/
│   ├── train/    # .txt files: class_id cx cy w h (normalized)
│   └── val/
└── data.yaml
```

In [ ]:
import shutil
from xml.etree import ElementTree as ET
from dataset import VOC_CLASSES

def convert_voc_to_yolo_format(voc_root, output_dir):
    """
    Convert PASCAL VOC XML annotations sang YOLO format (.txt).
    YOLO format: class_id center_x center_y width height (all normalized 0-1)
    """
    os.makedirs(output_dir, exist_ok=True)

    for split in ["train", "val"]:
        img_out = os.path.join(output_dir, "images", split)
        lbl_out = os.path.join(output_dir, "labels", split)
        os.makedirs(img_out, exist_ok=True)
        os.makedirs(lbl_out, exist_ok=True)

        # Doc danh sach image IDs
        split_file = os.path.join(voc_root, "ImageSets", "Main", f"{split}.txt")
        with open(split_file, "r") as f:
            image_ids = [line.strip() for line in f.readlines() if line.strip()]

        print(f"Converting {split}: {len(image_ids)} images...")

        for img_id in tqdm(image_ids, desc=split):
            # Copy image
            src_img = os.path.join(voc_root, "JPEGImages", f"{img_id}.jpg")
            dst_img = os.path.join(img_out, f"{img_id}.jpg")
            if not os.path.exists(dst_img):
                shutil.copy2(src_img, dst_img)

            # Convert annotation
            ann_path = os.path.join(voc_root, "Annotations", f"{img_id}.xml")
            tree = ET.parse(ann_path)
            root = tree.getroot()

            size = root.find("size")
            img_w = float(size.find("width").text)
            img_h = float(size.find("height").text)

            label_lines = []
            for obj in root.findall("object"):
                cls_name = obj.find("name").text.strip()
                if cls_name not in VOC_CLASSES:
                    continue

                cls_id = VOC_CLASSES.index(cls_name)  # 0-indexed cho YOLO
                bbox = obj.find("bndbox")
                xmin = float(bbox.find("xmin").text)
                ymin = float(bbox.find("ymin").text)
                xmax = float(bbox.find("xmax").text)
                ymax = float(bbox.find("ymax").text)

                # Convert sang YOLO format (normalized center_x, center_y, w, h)
                cx = (xmin + xmax) / 2.0 / img_w
                cy = (ymin + ymax) / 2.0 / img_h
                w = (xmax - xmin) / img_w
                h = (ymax - ymin) / img_h

                label_lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

            # Ghi file label
            lbl_path = os.path.join(lbl_out, f"{img_id}.txt")
            with open(lbl_path, "w") as f:
                f.write("\n".join(label_lines))

    # Tao data.yaml
    yaml_content = f"""path: {os.path.abspath(output_dir)}
train: images/train
val: images/val

nc: {len(VOC_CLASSES)}
names: {VOC_CLASSES}
"""
    yaml_path = os.path.join(output_dir, "data.yaml")
    with open(yaml_path, "w") as f:
        f.write(yaml_content)

    print(f"\ndata.yaml saved to {yaml_path}")
    return yaml_path


# Convert dataset
voc_root = os.path.join(DATA_ROOT, "VOCdevkit/VOC2012")
yolo_dataset_dir = os.path.join(DATA_ROOT, "VOC_YOLO")
data_yaml = convert_voc_to_yolo_format(voc_root, yolo_dataset_dir)
print(f"\nYOLO dataset ready at: {yolo_dataset_dir}")

## 3.3 Fine-tune YOLOv3 tren VOC

### Hyperparameters:
- **Image size**: 640 (mac dinh cua Ultralytics, YOLOv3 goc dung 416)
- **Batch size**: 16
- **Epochs**: 30 (YOLO can nhieu epoch hon do one-stage)
- **Optimizer**: SGD (mac dinh cua Ultralytics)
- **Augmentation**: Mosaic, HSV, Flip (built-in)

In [ ]:
# Fine-tune YOLOv3 tren VOC
results = model.train(
    data=data_yaml,
    epochs=30,
    imgsz=640,
    batch=16,
    name="yolov3_voc",
    project=RESULTS_DIR,
    pretrained=True,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,        # Final LR = lr0 * lrf
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    # Augmentation
    mosaic=1.0,
    flipud=0.0,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    # Save
    save=True,
    save_period=5,
    val=True,
    verbose=True,
)

## 3.4 Danh gia YOLOv3

In [ ]:
# Load best model va danh gia
best_model_path = os.path.join(RESULTS_DIR, "yolov3_voc", "weights", "best.pt")
best_model = YOLO(best_model_path)

# Evaluate tren val set
val_results = best_model.val(data=data_yaml, imgsz=640, batch=16, verbose=True)

print(f"\n{'='*60}")
print(f"  YOLOv3 - Evaluation Results")
print(f"{'='*60}")
print(f"  mAP@0.5:      {val_results.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {val_results.box.map:.4f}")
print(f"  Precision:     {val_results.box.mp:.4f}")
print(f"  Recall:        {val_results.box.mr:.4f}")
print(f"{'='*60}")

# Per-class AP
print(f"\n  {'Class':<15} {'AP@0.5':>8}")
print(f"  {'-'*25}")
for i, cls in enumerate(VOC_CLASSES):
    if i < len(val_results.box.ap50):
        print(f"  {cls:<15} {val_results.box.ap50[i]:.4f}")

## 3.5 Do Inference Speed

In [ ]:
# Do inference speed
import glob
from PIL import Image

val_images = glob.glob(os.path.join(DATA_ROOT, "VOC_YOLO/images/val/*.jpg"))[:100]

times = []
for img_path in tqdm(val_images, desc="Measuring speed"):
    start = time.time()
    _ = best_model.predict(img_path, imgsz=640, verbose=False)
    times.append(time.time() - start)

# Bo 10 warmup dau
times = times[10:]
avg_time = np.mean(times)
fps = 1.0 / avg_time

print(f"\nYOLOv3 Inference Speed:")
print(f"  Average time: {avg_time * 1000:.1f} ms/image")
print(f"  FPS: {fps:.1f}")

# Save ket qua
yolov3_results = {
    "mAP_50": float(val_results.box.map50),
    "mAP_50_95": float(val_results.box.map),
    "precision": float(val_results.box.mp),
    "recall": float(val_results.box.mr),
    "fps": float(fps),
    "avg_inference_ms": float(avg_time * 1000),
    "total_params": sum(p.numel() for p in best_model.model.parameters()),
    "per_class": {
        cls: {"ap": float(val_results.box.ap50[i]) if i < len(val_results.box.ap50) else 0}
        for i, cls in enumerate(VOC_CLASSES)
    },
}

with open(os.path.join(RESULTS_DIR, "eval_results.json"), "w") as f:
    json.dump(yolov3_results, f, indent=2)
print(f"Results saved.")

## 3.6 Visualization - Detection Results

In [ ]:
# Hien thi ket qua detection
np.random.seed(123)
sample_images = np.random.choice(val_images, 6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(20, 14))
for ax, img_path in zip(axes.flat, sample_images):
    # Predict
    result = best_model.predict(img_path, imgsz=640, conf=0.5, verbose=False)[0]
    
    # Load va hien thi anh goc
    img = Image.open(img_path)
    ax.imshow(np.array(img))
    
    import matplotlib.patches as patches
    
    # Ve predictions
    if result.boxes is not None:
        for box, cls, conf in zip(result.boxes.xyxy.cpu(), result.boxes.cls.cpu(), result.boxes.conf.cpu()):
            x1, y1, x2, y2 = box.numpy()
            cls_name = VOC_CLASSES[int(cls)]
            rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                      edgecolor="lime", facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, y1-3, f"{cls_name}:{conf:.2f}", fontsize=7, color="white",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="green", alpha=0.8))
    
    n_det = len(result.boxes) if result.boxes is not None else 0
    ax.set_title(f"{n_det} detections", fontsize=10)
    ax.axis("off")

plt.suptitle("YOLOv3 Detection Results (conf >= 0.5)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "detection_samples.png"), dpi=150, bbox_inches="tight")
plt.show()

## 3.7 Phan tich YOLOv3

### Diem manh:
- **Nhanh hon nhieu so voi Faster R-CNN**: One-stage approach, chi 1 lan forward pass
- **Multi-scale detection**: 3 scales (13x13, 26x26, 52x52) giup detect objects nhieu kich thuoc
- **Darknet-53 backbone**: Manh hon VGG, nhe hon ResNet, su dung residual connections
- **Global context**: Nhin toan bo image -> it false positives tu background

### Diem yeu:
- **Kem chinh xac hon Faster R-CNN** (thuong): Khong co buoc refine bbox nhu two-stage
- **Anchor-based**: Van phu thuoc vao anchor boxes, can chon anchor phu hop cho dataset
- **Loss function phuc tap**: Can can bang giua objectness, classification, va regression loss

### So voi YOLOv1:
1. **Multi-scale** (v1 chi predict o 1 scale)
2. **Anchor boxes** (v1 predict truc tiep, khong co anchor)
3. **Darknet-53** (v1 dung GoogLeNet-inspired network)
4. **Multi-label classification** (v1 dung softmax, v3 dung binary cross-entropy)

### Key insight:
YOLOv3 sacrifice mot chut accuracy de doi lay toc do gap 5-6 lan. Trong nhieu ung dung thuc te (self-driving, surveillance), toc do quan trong hon do chinh xac tuyet doi.